# Detecting Topic Merges and Splits in Dynamic Political Conversations

Cláudia Oliveira 

Supervisor - Prof. Dr. Álvaro Figueira

Faculty of Science, University of Porto

### UN Debates

In [ ]:
import tomotopy as tp
import pandas as pd
import ast
import os
os.chdir("../..")

# ======================================
# 1. Load CSV
# ======================================

df = pd.read_csv("./datasets/undebates/UN_General_Debates_Tokens.csv")
#df = pd.read_csv("./datasets/stateofunion_tokens.csv")

df["Tokens"] = df["Tokens"].apply(ast.literal_eval)

# ======================================
# 2. Create timepoints using the Year column
# ======================================
# Tomotopy requires timepoints as integers 0...(T-1)

df["timepoint"] = df["Year"].astype("category").cat.codes

# Number of distinct timepoints
num_periods = df["timepoint"].max() + 1
print(f"Number of distinct years (timepoints): {num_periods}")

# ======================================
# 3. Data feeder for tomotopy
# ======================================
def df_data_feeder(df):
    for _, row in df.iterrows():
        text = " ".join(row["Tokens"])  # convert list to string
        yield text, None, {"timepoint": int(row["timepoint"])}

# ======================================
# 4. Create corpus
# ======================================
corpus = tp.utils.Corpus(tokenizer=tp.utils.SimpleTokenizer())
corpus.process(df_data_feeder(df))

print(f"Documents loaded: {len(corpus)}")

# ======================================
# 5. Create and train the DTM model
# ======================================
mdl = tp.DTModel(
    min_cf=10,        # minimum corpus frequency
    k=50,             # number of topics
    t=num_periods,    # number of timepoints
    phi_var=1e-2,
    corpus=corpus
)

mdl.train(0)
print("Model initialized.")

mdl.train(1000, show_progress=True)
mdl.summary()

# =========================================================
# 4. Get top words per topic per year
# =========================================================

#def print_top_words_by_topic(mdl, top_n=10):
#    for k in range(mdl.k):
#        print(f"\n===== Topic {k} =====")
#        for t in range(mdl.num_timepoints):
#            words = mdl.get_topic_words(k, timepoint=t, top_n=top_n)
#            print(f"  Year {t}: ", [w for w, _ in words])

#print_top_words_by_topic(mdl)

Number of distinct years (timepoints): 46
Documents loaded: 26362
Model initialized.


Iteration: 100%|██████████| 1000/1000 [42:44<00:00,  2.56s/it, LLPW: -2.178865]


<Basic Info>
| DTModel (current version: 0.13.0)
| 26362 docs, 6896627 words
| Total Vocabs: 25372, Used Vocabs: 25163
| Entropy of words: 8.24567
| Entropy of term-weighted words: 8.24567
| Removed Vocabs: <NA>
|
<Training Info>
| Iterations: 1000, Burn-in steps: 0
| Optimization Interval: 10
| Log-likelihood per word: -2.17687
|
<Initial Parameters>
| tw: TermWeight.ONE
| min_cf: 10 (minimum collection frequency of words)
| min_df: 0 (minimum document frequency of words)
| rm_top: 0 (the number of top words to be removed)
| k: 50 (the number of topics between 1 ~ 32767)
| t: 46 (the number of timpoints)
| alpha_var: 0.1 (transition variance of alpha (per-document topic distribution))
| eta_var: 0.1 (variance of eta (topic distribution of each document) from its alpha )
| phi_var: 0.01 (transition variance of phi (word distribution of each topic))
| lr_a: 0.01 (shape parameter `a` greater than zero, for SGLD step size calculated as `e_i = a * (b + i) ^ (-c)`)
| lr_b: 0.1 (shape parame

In [ ]:
import numpy as np
from gensim.models import CoherenceModel
from gensim import corpora
from functions import *

# ---------------------------
# Topic diversity 
# ---------------------------
def evaluate_timepoint(mdl, df, timepoint, top_n=10):
    texts = df[df["timepoint"] == timepoint]["Tokens"].tolist()

    if len(texts) == 0:
        return np.nan, np.nan, np.nan, np.nan, np.nan

    topics_words = []
    for k in range(mdl.k):
        words = [
            w for w, _ in mdl.get_topic_words(
                k, timepoint=timepoint, top_n=top_n
            )
        ]
        topics_words.append(words)

    dictionary = corpora.Dictionary(texts)

    cv = CoherenceModel(
        topics=topics_words,
        texts=texts,
        dictionary=dictionary,
        coherence="c_v"
    ).get_coherence_per_topic()

    npmi = CoherenceModel(
        topics=topics_words,
        texts=texts,
        dictionary=dictionary,
        coherence="c_npmi"
    ).get_coherence_per_topic()

    umass = CoherenceModel(
        topics=topics_words,
        texts=texts,
        dictionary=dictionary,
        coherence="u_mass"
    ).get_coherence_per_topic()

    mean_cv = float(np.mean(cv))
    mean_npmi = float(np.mean(npmi))
    mean_umass = float(np.mean(umass))
    diversity = topic_diversity(topics_words)

    # Topic Quality
    tq = mean_cv * diversity

    return mean_cv, mean_npmi, mean_umass, diversity, tq


# ---------------------------
# Map timepoint -> year
# ---------------------------
timepoint_to_year = (
    df[["Year", "timepoint"]]
    .drop_duplicates()
    .set_index("timepoint")["Year"]
    .to_dict()
)


results = []

for t in range(num_periods):
    cv, npmi, umass, diversity, tq = evaluate_timepoint(mdl, df, t)

    results.append({
        "year": int(timepoint_to_year[t]),
        "cv": cv,
        "npmi": npmi,
        "umass": umass,
        "diversity": diversity,
        "tq": tq
    })

results_df = (
    pd.DataFrame(results)
    .sort_values("year")
    .reset_index(drop=True)
)

print(results_df.head())

   year        cv      npmi     umass  diversity        tq
0  1970  0.311083 -0.051783 -0.271041   0.661061  0.205645
1  1971  0.359869 -0.029077 -0.261328   0.520735  0.187396
2  1972  0.344086 -0.031219 -0.214844   0.557714  0.191902
3  1973  0.342299 -0.031822 -0.360562   0.533469  0.182606
4  1974  0.330868 -0.040109 -0.178448   0.573143  0.189635


In [3]:
results_df.to_csv("./datasets/un_tomotopydtm_qualityscore.csv", index=False)

EDGES 

In [ ]:
from sklearn.preprocessing import normalize

K = mdl.k
T = mdl.num_timepoints
top_n = 50  

# beta: K x T x top_n
beta = np.zeros((K, T, top_n))
for k in range(K):
    for t in range(T):
        words_probs = mdl.get_topic_words(k, timepoint=t, top_n=top_n)
        beta[k, t, :] = [p for w, p in words_probs]

beta_norm = normalize(beta.reshape(-1, top_n), axis=1).reshape(K, T, top_n)


edges = []
for k in range(K):
    for t in range(T-1):
        vec1 = beta_norm[k, t, :].reshape(1, -1)
        vec2 = beta_norm[k, t+1, :].reshape(1, -1)
        sim = cosine_similarity(vec1, vec2)[0, 0]
        edges.append({
            "topic": k,
            "timepoint1": t,
            "timepoint2": t+1,
            "similarity": sim
        })

df_edges = pd.DataFrame(edges)

In [13]:
df_edges.to_csv("datasets/undebates_dtm_relations.csv", index=False)

### State of Union

In [14]:
import tomotopy as tp
import pandas as pd
import ast

# ======================================
# 1. Load CSV
# ======================================

df = pd.read_csv("./datasets/stateofunion_tokens.csv")

# Convert string "['x','y']" → actual Python list
df["Tokens"] = df["Tokens"].apply(ast.literal_eval)

# ======================================
# 2. Create timepoints using the Year column
# ======================================
# Tomotopy requires timepoints as integers 0...(T-1)

df["timepoint"] = df["Year"].astype("category").cat.codes

# Number of distinct timepoints
num_periods = df["timepoint"].max() + 1
print(f"Number of distinct years (timepoints): {num_periods}")

# ======================================
# 3. Data feeder for tomotopy
# ======================================
def df_data_feeder(df):
    for _, row in df.iterrows():
        text = " ".join(row["Tokens"])  # convert list to string
        yield text, None, {"timepoint": int(row["timepoint"])}

# ======================================
# 4. Create corpus
# ======================================
corpus = tp.utils.Corpus(tokenizer=tp.utils.SimpleTokenizer())
corpus.process(df_data_feeder(df))

print(f"Documents loaded: {len(corpus)}")

# ======================================
# 5. Create and train the DTM model
# ======================================
mdl = tp.DTModel(
    min_cf=10,        # minimum corpus frequency
    k=50,             # number of topics
    t=num_periods,    # number of timepoints
    phi_var=1e-2,
    corpus=corpus
)

mdl.train(0)
print("Model initialized.")

mdl.train(1000, show_progress=True)
mdl.summary()

Number of distinct years (timepoints): 227
Documents loaded: 19283
Model initialized.


Iteration: 100%|██████████| 1000/1000 [38:17<00:00,  2.30s/it, LLPW: 148.472758]


<Basic Info>
| DTModel (current version: 0.13.0)
| 19283 docs, 540316 words
| Total Vocabs: 17984, Used Vocabs: 5550
| Entropy of words: 7.87793
| Entropy of term-weighted words: 7.87793
| Removed Vocabs: <NA>
|
<Training Info>
| Iterations: 1000, Burn-in steps: 0
| Optimization Interval: 10
| Log-likelihood per word: 148.47298
|
<Initial Parameters>
| tw: TermWeight.ONE
| min_cf: 10 (minimum collection frequency of words)
| min_df: 0 (minimum document frequency of words)
| rm_top: 0 (the number of top words to be removed)
| k: 50 (the number of topics between 1 ~ 32767)
| t: 227 (the number of timpoints)
| alpha_var: 0.1 (transition variance of alpha (per-document topic distribution))
| eta_var: 0.1 (variance of eta (topic distribution of each document) from its alpha )
| phi_var: 0.01 (transition variance of phi (word distribution of each topic))
| lr_a: 0.01 (shape parameter `a` greater than zero, for SGLD step size calculated as `e_i = a * (b + i) ^ (-c)`)
| lr_b: 0.1 (shape parame

In [ ]:
import numpy as np
from gensim.models import CoherenceModel
from gensim import corpora
from functions import *

# ---------------------------
# Topic diversity
# ---------------------------
def evaluate_timepoint(mdl, df, timepoint, top_n=10):
    texts = df[df["timepoint"] == timepoint]["Tokens"].tolist()

    if len(texts) == 0:
        return np.nan, np.nan, np.nan, np.nan, np.nan

    topics_words = []
    for k in range(mdl.k):
        words = [
            w for w, _ in mdl.get_topic_words(
                k, timepoint=timepoint, top_n=top_n
            )
        ]
        topics_words.append(words)

    dictionary = corpora.Dictionary(texts)

    cv = CoherenceModel(
        topics=topics_words,
        texts=texts,
        dictionary=dictionary,
        coherence="c_v"
    ).get_coherence_per_topic()

    npmi = CoherenceModel(
        topics=topics_words,
        texts=texts,
        dictionary=dictionary,
        coherence="c_npmi"
    ).get_coherence_per_topic()

    umass = CoherenceModel(
        topics=topics_words,
        texts=texts,
        dictionary=dictionary,
        coherence="u_mass"
    ).get_coherence_per_topic()

    mean_cv = float(np.mean(cv))
    mean_npmi = float(np.mean(npmi))
    mean_umass = float(np.mean(umass))
    diversity = topic_diversity(topics_words)

    # Topic Quality
    tq = mean_cv * diversity

    return mean_cv, mean_npmi, mean_umass, diversity, tq


# ---------------------------
# Map timepoint -> year
# ---------------------------
timepoint_to_year = (
    df[["Year", "timepoint"]]
    .drop_duplicates()
    .set_index("timepoint")["Year"]
    .to_dict()
)


results = []

for t in range(num_periods):
    cv, npmi, umass, diversity, tq = evaluate_timepoint(mdl, df, t)

    results.append({
        "year": int(timepoint_to_year[t]),
        "cv": cv,
        "npmi": npmi,
        "umass": umass,
        "diversity": diversity,
        "tq": tq
    })

results_df = (
    pd.DataFrame(results)
    .sort_values("year")
    .reset_index(drop=True)
)

print(results_df.head())

results_df.to_csv("./datasets/stateofunion_tomotopydtm_qualityscore.csv", index=False)

   year        cv      npmi      umass  diversity        tq
0  1791  0.582479 -0.624262 -21.203371   0.980327  0.571020
1  1792  0.567272 -0.620257 -21.139988   0.978612  0.555139
2  1793  0.575199 -0.647475 -22.057361   0.989469  0.569141
3  1794  0.613117 -0.636097 -22.173412   0.988980  0.606361
4  1795  0.575082 -0.622633 -21.622226   0.984327  0.566069


In [ ]:
from sklearn.preprocessing import normalize

K = mdl.k
T = mdl.num_timepoints
top_n = 50  

# beta: K x T x top_n
beta = np.zeros((K, T, top_n))
for k in range(K):
    for t in range(T):
        words_probs = mdl.get_topic_words(k, timepoint=t, top_n=top_n)
        beta[k, t, :] = [p for w, p in words_probs]

beta_norm = normalize(beta.reshape(-1, top_n), axis=1).reshape(K, T, top_n)


edges = []
for k in range(K):
    for t in range(T-1):
        vec1 = beta_norm[k, t, :].reshape(1, -1)
        vec2 = beta_norm[k, t+1, :].reshape(1, -1)
        sim = cosine_similarity(vec1, vec2)[0, 0]
        edges.append({
            "topic": k,
            "timepoint1": t,
            "timepoint2": t+1,
            "similarity": sim
        })

df_edges = pd.DataFrame(edges)

df_edges.to_csv("datasets/stateofunion_dtm_relations.csv", index=False)